# 01 — TrackMate preprocessing

This notebook imports TrackMate CSV files, extracts metadata, performs basic quality control, removes edge artifacts, filters short trajectories, and saves clean preprocessed tables for downstream diffusion and velocity analysis.

This notebook does **not** estimate diffusion coefficients and does **not** perform biological interpretation. Those steps are done in downstream notebooks.

## 1. Imports and configuration

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


plt.style.use("default")


# Repository root. If this notebook is located in notebooks/,
# Path.cwd() may point either to the repository root or to notebooks/.
CURRENT_DIR = Path.cwd()
PROJECT_DIR = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

DATA_DIR = PROJECT_DIR / "data"
RAW_DATA_DIR = DATA_DIR / "demo_data"
METADATA_DIR = DATA_DIR / "metadata"

RESULTS_DIR = PROJECT_DIR / "results"
PREPROCESSED_DIR = RESULTS_DIR / "preprocessed"
QC_DIR = RESULTS_DIR / "qc"

FIGURES_DIR = PROJECT_DIR / "figures"
QC_FIGURES_DIR = FIGURES_DIR / "qc"

for directory in [
    DATA_DIR,
    RAW_DATA_DIR,
    METADATA_DIR,
    RESULTS_DIR,
    PREPROCESSED_DIR,
    QC_DIR,
    FIGURES_DIR,
    QC_FIGURES_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


PROTEINS = ["Syx", "SNAP25"]

CONDITION_ORDER = [
    "control",
    "500uM_H2O2",
    "500uM_H2O2_30min",
]

CONDITION_LABELS = {
    "control": "Control",
    "500uM_H2O2": "500 µM H₂O₂",
    "500uM_H2O2_30min": "500 µM H₂O₂ 30 min",
}

# Core preprocessing parameters.
MIN_TRACK_LENGTH = 10
EDGE_MARGIN_UM = 0.5

# Optional: if TrackMate exports time in frames only, this value is used.
FRAME_INTERVAL_S = 0.02

print("Project directory:", PROJECT_DIR)
print("Raw data directory:", RAW_DATA_DIR)
print("Configuration loaded")

## 2. Expected input data structure

Recommended input structure:

```text
data/demo_data/
├── Syx/
│   ├── control/
│   │   ├── Sample_1/
│   │   └── Sample_2/
│   ├── 500uM_H2O2/
│   └── 500uM_H2O2_30min/
└── SNAP25/
    ├── control/
    ├── 500uM_H2O2/
    └── 500uM_H2O2_30min/
```

The notebook expects TrackMate `*_spots.csv` files. Large raw microscopy files (`*.nd2`, `*.tif`, etc.) should not be stored in the repository.

## 3. Utility functions

In [ ]:
def standardize_column_name(column_name):
    """Convert TrackMate column names to a compact snake-case style."""
    column_name = str(column_name).strip()
    column_name = column_name.replace(" ", "_")
    column_name = column_name.replace("-", "_")
    column_name = column_name.replace("/", "_")
    column_name = re.sub(r"[^A-Za-z0-9_]+", "", column_name)
    column_name = re.sub(r"_+", "_", column_name)
    return column_name.upper()


def normalize_condition_name(text):
    """Normalize condition names found in folder or file names."""
    text = str(text).lower()
    text = text.replace(" ", "_").replace("-", "_")

    if "30" in text and "h2o2" in text:
        return "500uM_H2O2_30min"
    if "500" in text and ("h2o2" in text or "ox" in text):
        return "500uM_H2O2"
    if "control" in text or "ctrl" in text:
        return "control"

    return "unknown"


def infer_protein(path):
    """Infer protein name from path parts or file name."""
    full_text = " ".join(path.parts).lower()

    if "snap" in full_text:
        return "SNAP25"
    if "syx" in full_text or "syntaxin" in full_text:
        return "Syx"

    return "unknown"


def infer_condition(path):
    """Infer condition from parent folders and file name."""
    for part in reversed(path.parts):
        condition = normalize_condition_name(part)
        if condition != "unknown":
            return condition
    return "unknown"


def infer_sample(path):
    """Infer sample number from folder or file name."""
    full_text = " ".join(path.parts)

    patterns = [
        r"Sample[_\s-]*(\d+)",
        r"sample[_\s-]*(\d+)",
        r"C(\d+)[-_]?cs",
        r"cs(\d+)",
    ]

    for pattern in patterns:
        match = re.search(pattern, full_text)
        if match:
            return int(match.group(1))

    return np.nan


def infer_cell(path):
    """Infer cell number from file name."""
    match = re.search(r"cell[_\s-]?(\d+)", path.name, flags=re.IGNORECASE)
    if match:
        return int(match.group(1))
    return np.nan


def parse_file_metadata(path):
    """Extract metadata from TrackMate CSV file path."""
    return {
        "protein": infer_protein(path),
        "condition": infer_condition(path),
        "sample": infer_sample(path),
        "cell": infer_cell(path),
        "file_name": path.name,
        "file_path": str(path),
    }

## 4. Find TrackMate CSV files

In [ ]:
spot_files = sorted(RAW_DATA_DIR.rglob("*_spots.csv"))

metadata = pd.DataFrame(
    [parse_file_metadata(path) for path in spot_files]
)

if metadata.empty:
    print("No *_spots.csv files were found.")
    print("Place TrackMate CSV files into:", RAW_DATA_DIR)
else:
    metadata = metadata.sort_values(
        ["protein", "sample", "cell", "condition", "file_name"],
        na_position="last",
    ).reset_index(drop=True)

metadata_path = METADATA_DIR / "trackmate_files_metadata.csv"
metadata.to_csv(metadata_path, index=False)

print(f"Found {len(metadata)} TrackMate spot files")
print("Saved metadata:", metadata_path)

metadata

## 5. TrackMate CSV reader

In [ ]:
COLUMN_ALIASES = {
    "TRACK_ID": "track_id",
    "POSITION_X": "x",
    "POSITION_Y": "y",
    "POSITION_T": "t",
    "FRAME": "frame",
    "QUALITY": "quality",
    "SNR_CH1": "snr",
    "SIGNAL_NOISE_RATIO_CH1": "snr",
    "MEAN_INTENSITY_CH1": "mean_intensity",
}


def read_trackmate_spots(file_path):
    """Read TrackMate spots CSV and return standardized columns.

    TrackMate CSV exports may contain several service rows after the header.
    This function reads the file, standardizes column names, and removes rows
    where required numeric fields cannot be parsed.
    """
    file_path = Path(file_path)

    raw = pd.read_csv(file_path, low_memory=False)
    raw.columns = [standardize_column_name(column) for column in raw.columns]

    rename_map = {
        column: COLUMN_ALIASES[column]
        for column in raw.columns
        if column in COLUMN_ALIASES
    }

    spots = raw.rename(columns=rename_map).copy()

    required_columns = ["track_id", "x", "y", "frame"]
    missing_columns = [
        column for column in required_columns
        if column not in spots.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing required columns in {file_path.name}: "
            f"{missing_columns}. Available columns: {list(spots.columns)}"
        )

    numeric_columns = [
        "track_id",
        "x",
        "y",
        "frame",
        "t",
        "quality",
        "snr",
        "mean_intensity",
    ]

    for column in numeric_columns:
        if column in spots.columns:
            spots[column] = pd.to_numeric(spots[column], errors="coerce")

    spots = spots.dropna(subset=required_columns).copy()

    spots["track_id"] = spots["track_id"].astype(int)
    spots["frame"] = spots["frame"].astype(int)

    if "t" not in spots.columns or spots["t"].isna().all():
        spots["t"] = spots["frame"] * FRAME_INTERVAL_S

    spots = spots.sort_values(["track_id", "frame"]).reset_index(drop=True)

    return spots

## 6. Edge filtering and track QC functions

In [ ]:
def get_track_bounds(spots):
    """Calculate spatial bounds and length for each trajectory."""
    bounds = (
        spots.groupby("track_id", sort=False)
        .agg(
            xmin=("x", "min"),
            xmax=("x", "max"),
            ymin=("y", "min"),
            ymax=("y", "max"),
            track_length=("frame", "count"),
            first_frame=("frame", "min"),
            last_frame=("frame", "max"),
        )
        .reset_index()
    )

    return bounds


def classify_edge_tracks(spots, margin_um=0.5):
    """Classify tracks as edge or non-edge based on track spatial bounds."""
    x_min_image = spots["x"].min()
    x_max_image = spots["x"].max()
    y_min_image = spots["y"].min()
    y_max_image = spots["y"].max()

    bounds = get_track_bounds(spots)

    edge_mask = (
        (bounds["xmin"] <= x_min_image + margin_um)
        | (bounds["xmax"] >= x_max_image - margin_um)
        | (bounds["ymin"] <= y_min_image + margin_um)
        | (bounds["ymax"] >= y_max_image - margin_um)
    )

    bounds["is_edge_track"] = edge_mask

    return bounds


def remove_edge_tracks(spots, margin_um=0.5):
    """Remove tracks touching the image border zone."""
    bounds = classify_edge_tracks(spots, margin_um=margin_um)

    edge_track_ids = bounds.loc[
        bounds["is_edge_track"],
        "track_id",
    ]

    filtered = spots[
        ~spots["track_id"].isin(edge_track_ids)
    ].copy()

    return filtered, bounds


def filter_short_tracks(spots, min_track_length=10):
    """Remove trajectories shorter than min_track_length."""
    track_lengths = spots.groupby("track_id").size()
    valid_track_ids = track_lengths[
        track_lengths >= min_track_length
    ].index

    filtered = spots[
        spots["track_id"].isin(valid_track_ids)
    ].copy()

    return filtered

## 7. QC plotting functions

In [ ]:
def setup_axis(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(alpha=0.2)


def plot_edge_filter_example(spots, bounds, title, output_path=None):
    """Visualize kept and removed localizations after edge filtering."""
    edge_ids = set(
        bounds.loc[bounds["is_edge_track"], "track_id"].to_numpy()
    )

    edge_spots = spots[spots["track_id"].isin(edge_ids)]
    kept_spots = spots[~spots["track_id"].isin(edge_ids)]

    fig, ax = plt.subplots(figsize=(5, 5))

    ax.scatter(
        kept_spots["x"],
        kept_spots["y"],
        s=2,
        alpha=0.25,
        label="kept tracks",
    )

    ax.scatter(
        edge_spots["x"],
        edge_spots["y"],
        s=2,
        alpha=0.25,
        label="removed edge tracks",
    )

    ax.set_aspect("equal")
    ax.set_xlabel("x, µm")
    ax.set_ylabel("y, µm")
    ax.set_title(title)
    ax.legend(frameon=False, loc="center")
    setup_axis(ax)
    plt.tight_layout()

    if output_path is not None:
        plt.savefig(output_path, dpi=300, bbox_inches="tight")

    plt.show()


def plot_track_length_distribution(spots, title, output_path=None):
    """Plot track length distribution before or after filtering."""
    track_lengths = spots.groupby("track_id").size()

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(track_lengths, bins=40)
    ax.axvline(
        MIN_TRACK_LENGTH,
        linestyle="--",
        linewidth=2,
        label=f"min length = {MIN_TRACK_LENGTH}",
    )

    ax.set_xlabel("Track length, frames")
    ax.set_ylabel("Number of tracks")
    ax.set_title(title)
    ax.legend(frameon=False)
    setup_axis(ax)
    plt.tight_layout()

    if output_path is not None:
        plt.savefig(output_path, dpi=300, bbox_inches="tight")

    plt.show()

## 8. Preprocess all TrackMate files

In [ ]:
all_raw_spots = []
all_filtered_spots = []
qc_rows = []

if metadata.empty:
    print("No files to preprocess. Add TrackMate *_spots.csv files and rerun the notebook.")
else:
    for _, row in metadata.iterrows():
        file_path = Path(row["file_path"])
        print("Processing:", file_path.name)

        spots_raw = read_trackmate_spots(file_path)
        spots_raw["protein"] = row["protein"]
        spots_raw["condition"] = row["condition"]
        spots_raw["sample"] = row["sample"]
        spots_raw["cell"] = row["cell"]
        spots_raw["file_name"] = row["file_name"]

        spots_no_edge, bounds = remove_edge_tracks(
            spots_raw,
            margin_um=EDGE_MARGIN_UM,
        )

        spots_filtered = filter_short_tracks(
            spots_no_edge,
            min_track_length=MIN_TRACK_LENGTH,
        )

        n_tracks_raw = spots_raw["track_id"].nunique()
        n_edge_tracks = int(bounds["is_edge_track"].sum())
        n_tracks_after_edge = spots_no_edge["track_id"].nunique()
        n_tracks_final = spots_filtered["track_id"].nunique()

        qc_rows.append({
            "protein": row["protein"],
            "condition": row["condition"],
            "sample": row["sample"],
            "cell": row["cell"],
            "file_name": row["file_name"],
            "n_spots_raw": len(spots_raw),
            "n_tracks_raw": n_tracks_raw,
            "n_edge_tracks_removed": n_edge_tracks,
            "edge_tracks_removed_percent": 100 * n_edge_tracks / n_tracks_raw if n_tracks_raw else np.nan,
            "n_tracks_after_edge": n_tracks_after_edge,
            "n_tracks_final": n_tracks_final,
            "edge_margin_um": EDGE_MARGIN_UM,
            "min_track_length": MIN_TRACK_LENGTH,
        })

        all_raw_spots.append(spots_raw)
        all_filtered_spots.append(spots_filtered)

    raw_spots = pd.concat(all_raw_spots, ignore_index=True)
    filtered_spots = pd.concat(all_filtered_spots, ignore_index=True)
    qc_summary = pd.DataFrame(qc_rows)

    raw_path = PREPROCESSED_DIR / "raw_spots_combined.csv"
    filtered_path = PREPROCESSED_DIR / "filtered_spots_combined.csv"
    qc_path = QC_DIR / "preprocessing_qc_summary.csv"

    raw_spots.to_csv(raw_path, index=False)
    filtered_spots.to_csv(filtered_path, index=False)
    qc_summary.to_csv(qc_path, index=False)

    print("Saved raw spots:", raw_path)
    print("Saved filtered spots:", filtered_path)
    print("Saved QC summary:", qc_path)

qc_summary if "qc_summary" in globals() else pd.DataFrame()

## 9. QC summary plots

In [ ]:
if "qc_summary" in globals() and not qc_summary.empty:
    fig, ax = plt.subplots(figsize=(7, 4))

    conditions = [
        condition for condition in CONDITION_ORDER
        if condition in qc_summary["condition"].unique()
    ]

    data = [
        qc_summary.loc[
            qc_summary["condition"] == condition,
            "edge_tracks_removed_percent",
        ].dropna().to_numpy()
        for condition in conditions
    ]

    ax.boxplot(data, labels=[CONDITION_LABELS[c] for c in conditions])
    ax.set_ylabel("Removed edge tracks, %")
    ax.set_title(f"Edge filtering impact, margin = {EDGE_MARGIN_UM} µm")
    setup_axis(ax)
    plt.xticks(rotation=20)
    plt.tight_layout()

    output_path = QC_FIGURES_DIR / "edge_filtering_impact.png"
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()

    print("Saved:", output_path)

In [ ]:
if "raw_spots" in globals() and not raw_spots.empty:
    example_file = raw_spots["file_name"].iloc[0]
    example_spots = raw_spots[raw_spots["file_name"] == example_file].copy()
    _, example_bounds = remove_edge_tracks(example_spots, margin_um=EDGE_MARGIN_UM)

    output_path = QC_FIGURES_DIR / "edge_filtering_example.png"

    plot_edge_filter_example(
        example_spots,
        example_bounds,
        title=f"Edge filter check\nmargin = {EDGE_MARGIN_UM} µm",
        output_path=output_path,
    )

    print("Saved:", output_path)

In [ ]:
if "filtered_spots" in globals() and not filtered_spots.empty:
    output_path = QC_FIGURES_DIR / "filtered_track_length_distribution.png"

    plot_track_length_distribution(
        filtered_spots,
        title="Filtered track length distribution",
        output_path=output_path,
    )

    print("Saved:", output_path)

## 10. Output files

This notebook generates the following files:

```text
data/metadata/trackmate_files_metadata.csv
results/preprocessed/raw_spots_combined.csv
results/preprocessed/filtered_spots_combined.csv
results/qc/preprocessing_qc_summary.csv
figures/qc/edge_filtering_impact.png
figures/qc/edge_filtering_example.png
figures/qc/filtered_track_length_distribution.png
```

These outputs are used by downstream notebooks for diffusion, velocity, mobility-state, and clustering analyses.